This notebook is for checking users and format the analysis cohort

In [1]:
import duckdb
from ga4case.path import SESSIONS, USERS

In [2]:
con = duckdb.connect()
con.sql(
    f"create or replace view sessions as select * from read_parquet('{SESSIONS.as_posix()}')"
)
con.sql(
    f"create or replace view users as select * from read_parquet('{USERS.as_posix()}')"
);

In [3]:
con.sql("select * from users limit 2").df()

,user_pseudo_id,first_session_number,first_visit_start,first_visit_end,landing_page_title,n_events,distinct_pages,active_seconds,viewed_product,searched,began_checkout,purchased_in_first_visit,saw_error_page,n_later_visits,next_visit_start,next_visit_page_view_start,later_purchase_start
0,1000823.8498711409,1,2021-01-18 02:57:43.062961,2021-01-18 02:59:06.554731,Home,8,2,74.644,0,0,0,0,0,1,2021-01-18 05:07:16.440316,NaT,NaT
1,1001406.4915130369,1,2021-01-28 13:58:41.575135,2021-01-28 13:58:59.223624,YouTube | Shop by Brand | Google Merchandise S...,6,1,15.674,0,0,0,0,0,0,NaT,NaT,NaT


In [4]:
# Check whether users number match sessions table
con.sql("""
    select
        'sessions' as source,
        count(distinct user_pseudo_id) as users,
        count(*) as visits,
        count(distinct user_pseudo_id) filter (where purchased = 1) as buyers
    from sessions
    union all
    select
        'users' as source,
        count(*) as users,
        sum(1 + n_later_visits)::bigint as visits,
        count(*) filter (where purchased_in_first_visit = 1 or later_purchase_start is not null) as buyers
    from users
""").df()

,source,users,visits,buyers
0,sessions,270154,360129,4419
1,users,270154,360129,4419


In [5]:
# Check the number of re-visit users
con.sql("""
    select
        count(*) as users,
        count(*) filter (where n_later_visits = 0) as only_one_visit,
        count(next_visit_start) as with_later_visit,
        count(next_visit_page_view_start) as with_later_visit_with_page_view,
        count(later_purchase_start) as with_later_purchase
    from users
    """).df()

,users,only_one_visit,with_later_visit,with_later_visit_with_page_view,with_later_purchase
0,270154,222790,47364,34756,2464


In [6]:
# Check the new users
con.sql("""
    select
        case when first_session_number = 1 then 'new user' else 'first visit before starting time' end as user_type,
        count(*) as users,
        round(100.0 * count(*) / sum(count(*)) over ( ), 1) as pct_users
    from users
    group by user_type
    order by users desc
""").df()

,user_type,users,pct_users
0,new user,260866,96.6
1,first visit before starting time,9288,3.4


In [7]:
# Check on user cohort suitable for analysis: excluded users not suitable for analysis
con.sql("""
    select
        count(*) as all_users,
        count(*) filter (where first_session_number > 1) as first_visit_before_starting_time,
        count(*) filter (where first_session_number = 1) as new_users,
        count(*) filter (where first_session_number = 1
                         and first_visit_end >= timestamp '2021-01-25') as less_than_7_days_follow_up,
        count(*) filter (where first_session_number = 1
                         and first_visit_end < timestamp '2021-01-25'
                         and purchased_in_first_visit = 1) as bought_in_first_visit,
        count(*) filter (where first_session_number = 1
                         and first_visit_end < timestamp '2021-01-25'
                         and purchased_in_first_visit = 0) as analysis_cohort
    from users
    """).df()

,all_users,first_visit_before_starting_time,new_users,less_than_7_days_follow_up,bought_in_first_visit,analysis_cohort
0,270154,9288,260866,19250,1678,239938


In [8]:
# How many users revist and purchase within 7 days
con.sql("""
    create or replace view cohort as
    select *
    from users
    where first_session_number = 1
    and first_visit_end < timestamp '2021-01-25'
    and purchased_in_first_visit = 0
""")

con.sql("""
    select 
        count(*) as users,
        round(100.0 * avg(coalesce(next_visit_start <= first_visit_end + interval 7 day, false):: int), 1) as pct_any_visit_7day,
        round(100.0 * avg(coalesce(next_visit_page_view_start <= first_visit_end + interval 7 day, false)::int), 1) as pct_visit_with_page_view_7day,
        round(100.0 * avg(coalesce(later_purchase_start <= first_visit_end + interval 7 day, false)::int), 2) as pct_purchase_7day
    from cohort
""").df()

,users,pct_any_visit_7day,pct_visit_with_page_view_7day,pct_purchase_7day
0,239938,14.4,9.6,0.48


In [9]:
# Check when will users revisit, any event happen?
con.sql("""
    with gaptime as (
        select
            date_diff('second', first_visit_end, next_visit_start) / 3600.0 as gap_hours,
            coalesce(next_visit_page_view_start = next_visit_start, false) as next_has_page_view
    from cohort
    where next_visit_start is not null
    )
    select
        case
            when gap_hours < 1 then '1: under 1 hour'
            when gap_hours < 24 then '2: 1 to 24 hours'
            when gap_hours < 24 * 7 then '3: 1 to 7 days'
            else '4: more than 7 days'
        end as time_to_next_visit,
        count(*) as users,
        round(100.0 * count(*) / sum(count(*)) over (), 1) as pct_users,
        round(100.0* avg(next_has_page_view::int), 1) as pct_next_visit_has_page_view
    from gaptime
    group by time_to_next_visit
    order by time_to_next_visit
""").df()

,time_to_next_visit,users,pct_users,pct_next_visit_has_page_view
0,1: under 1 hour,6282,15.9,49.5
1,2: 1 to 24 hours,20526,52.0,57.2
2,3: 1 to 7 days,7835,19.8,80.9
3,4: more than 7 days,4857,12.3,96.5


Finding summary:
- Users table matches sessions table: users, visits and purchasers
- 239,938 were selected for analysis cohort
- 67.9% revisit happens within 24 hours and around half of them does not have page view
- New user =  the first visit in the data with session number 1: 260,866 users and 9,288 excluded
- Come back = a later revisit with a page view  within 7 days after the first visit: accounts for 9.6% 
    - A later revisit accounts for 14.4% but some of them do not have any activity, thus not useful
- Purchase later = a later visit within 7 days and ended with a purchase: Only 0.48%